In [ ]:
import pandas as pd
import numpy as np

PATH = "/content/env_impact_data.csv"

df = pd.read_csv(PATH)

# Clean column names (strip whitespace/newlines)
df.columns = df.columns.str.strip()

df.head()


,Country,Year,D_Expenditure_GDP,Environmental_impact(CO2e/capita),"PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)"
0,USA,1994,4.215265,19.83,13.78
1,USA,1995,3.860246,19.79,13.67
2,USA,1996,3.554982,20.14,13.52
3,USA,1997,3.554982,20.90,13.34
4,USA,1998,3.201558,20.83,13.13


In [4]:
NUMERIC_COLS = [
    "D_Expenditure_GDP",
    "Environmental_impact(CO2e/capita)",
    "PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)"
]

def to_numeric(series):
    return pd.to_numeric(
        series.astype(str)
              .str.strip()
              .replace({"--": np.nan, "nan": np.nan}),
        errors="coerce"
    )

for col in NUMERIC_COLS:
    df[col] = to_numeric(df[col])


In [5]:
df = df.sort_values(["Country", "Year"]).reset_index(drop=True)


In [6]:
missing_summary = (
    df.groupby("Country")[NUMERIC_COLS]
      .apply(lambda x: x.isna().sum())
)

print(missing_summary)


         D_Expenditure_GDP  Environmental_impact(CO2e/capita)  \
Country                                                         
CHN                      0                                  1   
FRA                      0                                  1   
GBR                      0                                  1   
RUS                      0                                  1   
USA                      0                                  1   

         PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)  
Country                                                                          
CHN                                                      4                       
FRA                                                      4                       
GBR                                                      4                       
RUS                                                      4                       
USA                                                 

In [7]:
def clean_co2e(group):
    s = group["Environmental_impact(CO2e/capita)"]
    s_interp = s.interpolate(method="linear", limit_direction="both")
    group["Environmental_impact(CO2e/capita)"] = s_interp
    return group

df = df.groupby("Country", group_keys=False).apply(clean_co2e)


In [8]:
def clean_pm25(group):
    s = group["PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)"]
    # Only interpolate short gaps (max 2 years)
    s_interp = s.interpolate(method="linear", limit=2)
    group["PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)"] = s_interp
    return group

df = df.groupby("Country", group_keys=False).apply(clean_pm25)


In [9]:
# Check remaining NaNs
df.isna().sum()


,0
Country,0
Year,0
D_Expenditure_GDP,0
Environmental_impact(CO2e/capita),0
"PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)",10


In [10]:
def prepare_country_ts(df, country, target):
    g = (
        df[df["Country"] == country]
        .sort_values("Year")
        .dropna(subset=[target, "D_Expenditure_GDP"])
        .set_index("Year")
    )
    return g


In [18]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score
)

# -----------------------------
# Load cleaned dataset
# -----------------------------
PATH = "/content/env_impact_data.csv"

df = pd.read_csv(PATH)
df.columns = df.columns.str.strip()

TARGET = "Environmental_impact(CO2e/capita)"
EXOG = "D_Expenditure_GDP"

def to_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.strip().replace({"--": np.nan}),
        errors="coerce"
    )

df[TARGET] = to_numeric(df[TARGET])
df[EXOG] = to_numeric(df[EXOG])

df = df.sort_values(["Country", "Year"]).reset_index(drop=True)

# -----------------------------
# Helper: 80/20 split per country
# -----------------------------
def time_split(g, train_frac=0.8):
    n = len(g)
    split = int(np.floor(n * train_frac))
    train = g.iloc[:split]
    test = g.iloc[split:]
    return train, test

# -----------------------------
# Evaluation container
# -----------------------------
rows = []

# -----------------------------
# Loop per country
# -----------------------------
for country, g in df.groupby("Country"):
    g = g.dropna(subset=[TARGET, EXOG]).sort_values("Year")

    if len(g) < 15:
        print(f"[SKIP] {country}: insufficient data")
        continue

    train, test = time_split(g, 0.8)

    y_train = train[TARGET]
    X_train = train[[EXOG]]

    y_test = test[TARGET]
    X_test = test[[EXOG]]

    # -----------------------------
    # ARIMAX model
    # (conservative order due to short series)
    # -----------------------------
    model = SARIMAX(
        y_train,
        exog=X_train,
        order=(1, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit(disp=False)

    # -----------------------------
    # Forecast on test set
    # -----------------------------
    forecast = results.get_forecast(
        steps=len(test),
        exog=X_test
    )

    y_pred = forecast.predicted_mean

    # -----------------------------
    # Regression metrics
    # -----------------------------
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # -----------------------------
    # Directional Precision & Recall
    # -----------------------------
    # True direction (actual)
    y_true_dir = (y_test.diff().dropna() > 0).astype(int)

    # Predicted direction
    y_pred_dir = (pd.Series(y_pred, index=y_test.index)
                    .diff()
                    .dropna() > 0).astype(int)

    # Align
    common_idx = y_true_dir.index.intersection(y_pred_dir.index)
    y_true_dir = y_true_dir.loc[common_idx]
    y_pred_dir = y_pred_dir.loc[common_idx]

    precision = precision_score(y_true_dir, y_pred_dir, zero_division=0)
    recall = recall_score(y_true_dir, y_pred_dir, zero_division=0)

    rows.append({
        "Country": country,
        "Train_years": len(train),
        "Test_years": len(test),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

# -----------------------------
# Results summary
# -----------------------------
results_df = pd.DataFrame(rows)
print("\n===== ARIMAX 80/20 Evaluation (per country) =====")
print(results_df.round(4).to_string(index=False))



===== ARIMAX 80/20 Evaluation (per country) =====
Country  Train_years  Test_years    MAE   RMSE      R2
    CHN           24           6 0.6322 0.7021 -2.3493
    FRA           24           6 0.1798 0.2335  0.2841
    GBR           24           6 0.8567 1.0049 -5.3633
    RUS           24           6 1.2945 1.5482 -4.3611
    USA           24           6 0.7474 0.8627 -0.5304
